In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from collections import defaultdict
import time
import numpy as np
import csv
import os
import random

In [ ]:
# Masked Label Ranking

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "../data/scidcc/SciDCC-masked-0.75.jsonl"
output_file = "multi-mw0.75-rank-llama-3b.jsonl"
frame_file = "multi-mw0.5-rank-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    ranking: list[str]

    
def generate_article(article_body, labels):
        
    prompt_rank = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {labels}
        
        Compare ALL labels from the list in PAIRS with each other (i.e., every possible combination) and evaluate, for each pair, which label describes the topic of article A better.   
        From these comparisons, derive a full ranking. 
        Output the labels in a JSON list in the order of their ranks, starting with the most relevant. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_rank}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.ranking
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY)
        return None

start = 0

masked = {}

with open(input_file, "r") as infile:
    for inline in infile:
        obj = json.loads(inline)
        art_title = obj["Title"]
        masked[art_title] = obj["masked_text"]


with open(frame_file, "r") as frame, open(output_file, "a") as outfile:
    for i, frameline in enumerate(frame):
        if i < start:
            continue

        frame_obj = json.loads(frameline)
        
        title = frame_obj["article_title:"]
        new_obj = copy.deepcopy(frame_obj)  # make a full copy
        
        if title in masked:
            labels = frame_obj["prediction"]["masked_multilabel_0.75"]
            if labels is None:
                new_obj["prediction"]["masked_ranking_0.75"] = None
            elif len(labels) < 2:
                new_obj["prediction"]["masked_ranking_0.75"] = labels
            new_obj["prediction"]["masked_ranking_0.75"] = generate_article(masked[title], labels)
    
        # write back as JSONL
        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Masked Label Selection

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


TOPICS = ["Earthquakes", "Pollution", "Genetically Modified", "Hurricanes Cyclones", "Agriculture & Food", "Animals", 
          "Weather", "Endangered Animals", "Climate", "Ozone Holes", "Biology", "New Species", "Environment", 
          "Biotechnology", "Geography", "Microbes", "Extinction", "Zoology", "Geology", "Global Warming"]


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "../data/scidcc/SciDCC-masked-0.75.jsonl"
output_file = "multi-mw0.75-select-llama-3b.jsonl"
frame_file = "multi-mw0.5-select-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    masked_multilabel: list[str]

    
def generate_article(article_body):
        
    prompt_multilabel = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {TOPICS}
        
        Read through the article and carefully analyse its contents. 
        Choose all labels that match the article's topic. There is no limit to the number of labels you may choose, as long as they are relevant to the article's subject(s). 
        You may also assign only one label if that is the only topic of the article. 
        Output ONLY the chosen labels in a JSON list. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_multilabel}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.masked_multilabel
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY + random.uniform(0, 3))
        return None

start = 0
masked = {}

with open(input_file, "r") as infile:
    for inline in infile:
        obj = json.loads(inline)
        art_title = obj["Title"]
        masked[art_title] = obj["masked_text"]


with open(frame_file, "r") as frame, open(output_file, "a") as outfile:
    
    for i, frameline in enumerate(frame):
        if i < start:
            continue

        frame_obj = json.loads(frameline)
        title = frame_obj["article_title:"]
        new_obj = copy.deepcopy(frame_obj)
        
        if title in masked:
            new_obj["prediction"]["masked_multilabel_0.75"] = generate_article(masked[title])
            #print(f"Processed article: {title}")
     
        # write back as JSONL
        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Paraphrased Label Ranking

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy
import time


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "../data/scidcc/SciDCC-pp.jsonl"
frame_file = "multi-pp-select-llama-3b.jsonl"
output_file = "multi-pp-rank-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    ranking: list[str]

    
def generate_article(article_body, labels):
        
    prompt_rank = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {labels}
        
        Compare ALL labels from the list in PAIRS with each other (i.e., every possible combination) and evaluate, for each pair, which label describes the topic of article A better.   
        From these comparisons, derive a full ranking. 
        Output the labels in a JSON list in the order of their ranks, starting with the most relevant. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_rank}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.ranking
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY)
        return None

start = 0

with open(input_file, "r") as infile, open(frame_file, "r") as frame, open(output_file, "a") as outfile:
    
    for i, (inline, frameline) in enumerate(zip(infile, frame)):
        if i < start:
            continue

        obj = json.loads(inline)
        frame_obj = json.loads(frameline)

        paraphrased_body = obj["body_variant"]
        labels = frame_obj["prediction"]["paraphrased_multilabel"]
        new_obj = copy.deepcopy(frame_obj)  # make a full copy
        
        if labels is None:
            new_obj["prediction"]["paraphrase_ranking"] = None
        elif len(labels) < 2:
            new_obj["prediction"]["paraphrase_ranking"] = labels
        new_obj["prediction"]["paraphrase_ranking"] = generate_article(paraphrased_body, labels)
    
        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Paraphrase Label Selection

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


TOPICS = ["Earthquakes", "Pollution", "Genetically Modified", "Hurricanes Cyclones", "Agriculture & Food", "Animals", 
          "Weather", "Endangered Animals", "Climate", "Ozone Holes", "Biology", "New Species", "Environment", 
          "Biotechnology", "Geography", "Microbes", "Extinction", "Zoology", "Geology", "Global Warming"]


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "../data/scidcc/SciDCC-pp.jsonl"
output_file = "multi-pp-select-llama-3b.jsonl"
frame_file = "multi-cs-rank-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    paraphrased_multilabel: list[str]

    
def generate_article(article_body):
        
    prompt_multilabel = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {TOPICS}
        
        Read through the article and carefully analyse its contents. 
        Choose all labels that match the article's topic. There is no limit to the number of labels you may choose, as long as they are relevant to the article's subject(s). 
        You may also assign only one label if that is the only topic of the article. 
        Output ONLY the chosen labels in a JSON list. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_multilabel}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.paraphrased_multilabel
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY + random.uniform(0, 3))
        return None

start = 0

with open(input_file, "r") as infile, open(frame_file, "r") as frame, open(output_file, "a") as outfile:
    
    for i, (inline, frameline) in enumerate(zip(infile, frame)):
        if i < start:
            continue

        obj = json.loads(inline)
        frame_obj = json.loads(frameline)

        paraphrased_body = obj["body_variant"]
        new_obj = copy.deepcopy(frame_obj)
        new_obj["prediction"]["paraphrased_multilabel"] = generate_article(paraphrased_body)
    
        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Code-Mixed Label Ranking

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "../data/scidcc/SciDCC-csw.jsonl"
frame_file = "multi-cs-select-llama-3b.jsonl"
output_file = "multi-cs-rank-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    ranking: list[str]

    
def generate_article(article_body, labels):
        
    prompt_rank = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        [Disclaimer: Ignore long sequences of repetitive tokens in the article, if any exist.]

        A: {article_body}
        
        L: {labels}
        
        Compare ALL labels from the list in PAIRS with each other (i.e., every possible combination) and evaluate, for each pair, which label describes the topic of article A better.   
        From these comparisons, derive a full ranking. 
        Output the labels in a JSON list in the order of their ranks, starting with the most relevant. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_rank}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.ranking
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY)
        return None

start = 0

with open(input_file, "r") as infile, open(frame_file, "r") as frame, open(output_file, "a") as outfile:
    
    for i, (inline, frameline) in enumerate(zip(infile, frame)):
        if i < start:
            continue

        obj = json.loads(inline)        
        frame_obj = json.loads(frameline)

        codeswitched_body = obj["codeswitched"]
        labels = frame_obj["prediction"]["codeswitched_multilabel"]
        
        new_obj = copy.deepcopy(frame_obj)  
        
        if labels is None:
            new_obj["prediction"]["codeswitch_ranking"] = None
        elif len(labels) < 2:
            new_obj["prediction"]["codeswitch_ranking"] = labels
        new_obj["prediction"]["codeswitch_ranking"] = generate_article(codeswitched_body, labels)
    
        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Code-Mixed Label Selection

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


TOPICS = ["Earthquakes", "Pollution", "Genetically Modified", "Hurricanes Cyclones", "Agriculture & Food", "Animals", 
          "Weather", "Endangered Animals", "Climate", "Ozone Holes", "Biology", "New Species", "Environment", 
          "Biotechnology", "Geography", "Microbes", "Extinction", "Zoology", "Geology", "Global Warming"]


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "../data/scidcc/SciDCC-csw.jsonl"
output_file = "multi-cs-select-llama-3b.jsonl"
frame_file = "multi-og-rank-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    codeswitched_multilabel: list[str]

    
def generate_article(article_body):
        
    prompt_multilabel = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {TOPICS}
        
        Read through the article and carefully analyse its contents. Ignore long sequences of repetitive tokens, if any exist. 
        Choose all labels that match the article's topic. There is no limit to the number of labels you may choose, as long as they are relevant to the article's subject(s). 
        You may also assign only one label if that is the only topic of the article. 
        Output ONLY the chosen labels in a JSON list. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_multilabel}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.codeswitched_multilabel
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY + random.uniform(0, 3))
        return None

start = 0

with open(input_file, "r") as infile, open(frame_file, "r") as frame, open(output_file, "a") as outfile:
    
    for i, (inline, frameline) in enumerate(zip(infile, frame)):
        if i < start:
            continue

        obj = json.loads(inline)        
        frame_obj = json.loads(frameline)

        codeswitched_body = obj["codeswitched"]
        new_obj = copy.deepcopy(frame_obj)
        new_obj["prediction"]["codeswitched_multilabel"] = generate_article(codeswitched_body)
    

        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Label Ranking

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "multi-og-select-llama-3b.jsonl"
output_file = "multi-og-rank-llama-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    ranking: list[str]

    
def generate_article(article_body, labels):
        
    prompt_rank = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {labels}
        
        Compare ALL labels from the list in PAIRS with each other (i.e., every possible combination) and evaluate, for each pair, which label describes the topic of article A better. 
        From these comparisons, derive a full ranking. 
        Output the labels in a JSON list in the order of their ranks, starting with the most relevant. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_rank}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.ranking
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY)
        return None

start = 0

with open(input_file, "r") as infile, open(output_file, "a") as outfile:
    
    for i, line in enumerate(infile):
        if i < start:
            continue

        obj = json.loads(line)       
        article_body = obj["article_body"]
        labels = obj["prediction"]["multi_label"]
        new_obj = copy.deepcopy(obj) 
        
 
        if labels is None:
            new_obj["prediction"]["ranking"] = None
        elif len(labels) < 2:
            new_obj["prediction"]["ranking"] = labels
        new_obj["prediction"]["ranking"] = generate_article(article_body, labels)
    

        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")


In [ ]:
# Label Selection

from ollama import Client
from ollama import chat
from pydantic import BaseModel
import json
import copy


TOPICS = ["Earthquakes", "Pollution", "Genetically Modified", "Hurricanes Cyclones", "Agriculture & Food", "Animals", 
          "Weather", "Endangered Animals", "Climate", "Ozone Holes", "Biology", "New Species", "Environment", 
          "Biotechnology", "Geography", "Microbes", "Extinction", "Zoology", "Geology", "Global Warming"]


client = Client(
    host='host',
)

model = 'llama3.2:3b'

input_file = "mistral-3B-single.jsonl"
output_file = "multi-og-select-mistral-3b.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

class StructuredOutput(BaseModel):
    body_multilabel: list[str]

    
def generate_article(article_body):
        
    prompt_multilabel = f"""
        You are a classifier. You are given an article (A) and a list of labels (L): 
        
        A: {article_body}
        
        L: {TOPICS}
        
        Read through the article and carefully analyse its contents. Choose all labels that match the article's topic. 
        There is no limit to the number of labels you may choose, as long as they are relevant to the article's subject(s). 
        You may also assign only one label if that is the only topic of the article. 
        Output ONLY the chosen labels in a JSON list. Do not produce any other output."""

    messages = [{'role': 'user','content': prompt_multilabel}]
    retries = 0
    while True:
        try:
            response = client.chat(model=model,
                                   messages=messages,
                                   options={"temperature":0.1,
                                            "seed":16,
                                            #"top_k":10,
                                            "top_p":1,
                                                },
                                    format=StructuredOutput.model_json_schema(),
                                    )
            predicted = StructuredOutput.model_validate_json(response.message.content)
            return predicted.body_multilabel
            
        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")

            if retries >= MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY + random.uniform(0, 3))
        return None

start = 0

with open(input_file, "r") as infile, open(output_file, "a") as outfile:
    
    for i, line in enumerate(infile):
        if i < start:
            continue

        obj = json.loads(line)        # parse JSON line → dict
        article_body = obj["article_body"]
        new_obj = copy.deepcopy(obj)  # make a full copy
        
        # add or modify fields
        new_obj["prediction"]["multi_label"] = generate_article(article_body)
    
        # write back as JSONL
        outfile.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
        outfile.flush()


print("All complete :)")
